# Tu primer scorecard con Nikodym

Este cuaderno construye un **scorecard de riesgo de crédito** de punta a punta y deja el
informe, el registro de auditoría y los libros de Excel listos para revisar. Las salidas
que ves abajo son **reales**: se generaron ejecutando este mismo cuaderno.

Lo que necesitas antes de empezar:

```bash
pip install "nikodym[scoring,report,excel,ui]"
```

Y nada más. No hay que configurar el motor: los valores de fábrica funcionan, y lo que sí
decide la institución —qué es un incumplimiento, qué muestra queda fuera de tiempo— se
pide de forma explícita.

Las rutas de las salidas se muestran relativas a la carpeta de este cuaderno.

## 1. Los datos

Usamos una cartera de consumo sintética que viene en el paquete (6.000 operaciones,
determinista), para que este cuaderno corra igual en cualquier máquina. En tu caso, `datos`
es la ruta de tu `.parquet` o `.csv`.

In [1]:
from pathlib import Path

import pandas as pd

from nikodym import Scorecard
from nikodym.ui.datasets import materialize

datos = materialize("consumo_comportamiento", workdir=Path("nikodym-runs"))

pd.read_parquet(datos).head()

,ingreso_mensual,deuda_ingreso,utilizacion_linea,mora_max_12m,antiguedad_meses,segmento,cohorte,bad_flag
loan_id,,,,,,,,
op-000000,332752.50,0.2874,0.4299,6,87,pensionado,2023Q2,0
op-000001,377467.40,0.4386,0.2493,6,84,pensionado,2023Q1,0
op-000002,1663079.61,0.1881,0.5301,6,3,asalariado,2024Q1,0
op-000003,472874.83,0.4824,0.1010,10,7,independiente,2024Q2,1
op-000004,321162.90,0.1270,0.5967,5,33,asalariado,2024Q1,1


## 2. La corrida completa

`Scorecard(...)` pide sólo lo institucional: cuál es la columna de incumplimiento, cuál
identifica la operación y cómo se separan las muestras. Todo lo demás —el esquema, qué
columnas son predictoras, cuáles son categóricas— **se infiere y se declara** en el
registro de auditoría.

`run()` corre las once etapas y **cada una cuenta lo que hizo** mientras avanza.

In [2]:
sc = Scorecard(
    data=datos,
    target="bad_flag",          # 1 = malo
    id="loan_id",
    cohort="cohorte",           # la añada de cada operación
    oot_cohorts=["2024Q2"],     # la muestra fuera de tiempo la decide la institución
    name="consumo_v01",
)
sc.run()

── Datos y muestras ──
Archivo: nikodym-runs\datasets\consumo_comportamiento.parquet (copia en nikodym-runs\consumo_v01\input\data-f7e0ff42e1fc1c9d.parquet)
6.000 filas · 1.407 malos (23,45 %) · 8 columnas
Muestras: Desarrollo 3.961 (23,33 %) · Holdout 1.031 (23,67 %) · Fuera de tiempo (OOT) 1.008 (23,71 %) — cohortes fuera de tiempo: 2024Q2 (columna «cohorte»); holdout 20 % del resto
Se infirió: esquema de 8 columnas (6 numéricas o de fecha, 2 de texto); 6 predictoras (1 categórica); fuera: cohorte (cohorte de la partición), bad_flag (define el incumplimiento, sería una fuga de información)
Identificador: loan_id (índice del archivo)
Evidencia de la corrida: nikodym-runs\consumo_v01\run
── Análisis exploratorio ──
Tasa de malos por cohorte: 5 períodos, media 23,33 %
Deterioro de la tasa en el tiempo: no evaluable (eje de cohorte, sin orden cronológico)
6 columnas descritas frente al incumplimiento
Marcas de calidad del archivo: casi única: ingreso_mensual
── Tramos y WoE ──
6 de 6 var

AUC en Fuera de tiempo (OOT),"0,656"
Gini en Fuera de tiempo (OOT),"0,312"
KS en Fuera de tiempo (OOT),"0,252"
Caída del AUC Desarrollo → Fuera de tiempo (OOT),"-0,056 (-7,9 %)"
Peor PSI entre score y PD,"0,0132 (Desarrollo vs. Holdout) → Estable"


## 3. Mirar una etapa por dentro

Cada etapa deja su resumen y su tabla de decisión. Aquí, la selección de variables: qué
entró, qué salió y **por qué**.

In [3]:
sc.results["selection"]

Variable,Entra,Motivo,IV,AUC,KS,Correlación máxima,VIF,IV Desarrollo,IV Holdout,IV Fuera de tiempo (OOT)
antiguedad_meses,sí,inclusión,"0,041","0,548","0,061",—,"1,00","0,041","0,109","0,101"
deuda_ingreso,sí,inclusión,"0,164","0,602","0,160",—,"1,00","0,164","0,107","0,170"
ingreso_mensual,sí,inclusión,"0,305","0,647","0,212",—,"1,00","0,305","0,252","0,140"
mora_max_12m,sí,inclusión,"0,022","0,540","0,064",—,"1,00","0,022","0,033","0,081"
segmento,no,IV insuficiente,"0,003","0,515","0,023",—,—,"0,003","0,009","0,006"
utilizacion_linea,sí,inclusión,"0,061","0,566","0,098",—,"1,00","0,061","0,219","0,085"


## 4. Una decisión humana

El modelador manda. `exclude(...)` —siempre con su `reason=`— escribe el config y queda en
el registro de auditoría con autor y motivo. `resume()` es una **corrida nueva y completa**
sobre el config vigente: la anterior queda al lado como respaldo, con su informe.

Aquí sacamos una variable que en la práctica no está disponible al originar el crédito.

In [4]:
sc.exclude("mora_max_12m", reason="no estará disponible al originar")
sc.resume()

Decisión registrada: exclude mora_max_12m — «no estará disponible al originar». Se aplica en la corrida siguiente: resume().
── Datos y muestras ──
Archivo: nikodym-runs\datasets\consumo_comportamiento.parquet (copia en nikodym-runs\consumo_v01\input\data-f7e0ff42e1fc1c9d.parquet)
6.000 filas · 1.407 malos (23,45 %) · 8 columnas
Muestras: Desarrollo 3.961 (23,33 %) · Holdout 1.031 (23,67 %) · Fuera de tiempo (OOT) 1.008 (23,71 %) — cohortes fuera de tiempo: 2024Q2 (columna «cohorte»); holdout 20 % del resto
Se infirió: esquema de 8 columnas (6 numéricas o de fecha, 2 de texto); 6 predictoras (1 categórica); fuera: cohorte (cohorte de la partición), bad_flag (define el incumplimiento, sería una fuga de información)
Identificador: loan_id (índice del archivo)
Evidencia de la corrida: nikodym-runs\consumo_v01\run
── Análisis exploratorio ──
Tasa de malos por cohorte: 5 períodos, media 23,33 %
Deterioro de la tasa en el tiempo: no evaluable (eje de cohorte, sin orden cronológico)
6 columna

AUC en Fuera de tiempo (OOT),"0,648"
Gini en Fuera de tiempo (OOT),"0,295"
KS en Fuera de tiempo (OOT),"0,232"
Caída del AUC Desarrollo → Fuera de tiempo (OOT),"-0,062 (-8,7 %)"
Peor PSI entre score y PD,"0,0172 (Desarrollo vs. OOT) → Estable"


## 5. El resumen final

Separa dos cosas que no son lo mismo: si la corrida **terminó** y si el modelo **pasa** la
validación técnica. Las cinco cifras clave se leen de un vistazo.

In [5]:
sc.summary()

AUC en Fuera de tiempo (OOT),"0,648"
Gini en Fuera de tiempo (OOT),"0,295"
KS en Fuera de tiempo (OOT),"0,232"
Caída del AUC Desarrollo → Fuera de tiempo (OOT),"-0,062 (-8,7 %)"
Peor PSI entre score y PD,"0,0172 (Desarrollo vs. OOT) → Estable"


## 6. Los entregables

La carpeta del proyecto queda con todo: el config completo que reproduce la corrida sin
este cuaderno, la copia de los datos que se leyeron, el registro de auditoría, el estudio
y el informe en HTML, Word y fuente editable.

`export_excel()` añade, si lo quieres, un libro por etapa con las mismas tablas del
informe.

In [6]:
print(sc.project_dir)
for libro in sc.export_excel():
    print(libro.name)

nikodym-runs\consumo_v01
Excel por etapa: 11 libros en nikodym-runs\consumo_v01\excel
01 Datos y muestras.xlsx
02 Análisis exploratorio.xlsx
03 Tramos y WoE.xlsx
04 Selección de variables.xlsx
05 Modelo.xlsx
06 Tarjeta de puntuación.xlsx
07 Calibración.xlsx
08 Desempeño.xlsx
09 Estabilidad.xlsx
10 Validación formal.xlsx
11 Decisiones.xlsx
